In [1]:
# Manipulação de dados
import pandas as pd

# Validação e construção de pipelines
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline


# Otimização de hiperparâmetros com Optuna
from optuna import create_study, logging
from optuna.distributions import (
    IntDistribution,
    FloatDistribution,
    CategoricalDistribution
)

# Análise de importância das features
from sklearn.inspection import permutation_importance
from sklearn.base import clone

# Modelos de regressão
from xgboost import XGBRegressor

# Métricas de avaliação
from sklearn.metrics import r2_score

c:\Users\Gabryel\Desktop\Projetos_ML\PriceMyHome\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
X_train = pd.read_csv(r"..\data\processed\V1\X_processed_train.csv")
X_test = pd.read_csv(r"..\data\processed\V1\X_processed_test.csv")
y_train = pd.read_csv(r"..\data\processed\V1\y_processed_train.csv")
y_test = pd.read_csv(r"..\data\processed\V1\y_processed_test.csv")
SEED = 42

In [5]:
def get_model(trial):

    model_name = trial.suggest_categorical(
        "model",
        ["xgb"]
    )

    if model_name == "xgb":
        return XGBRegressor(
            max_depth=trial.suggest_int("xgb_depth", 4, 12),
            learning_rate=trial.suggest_float("xgb_lr", 1e-3, 0.3, log=True),
            n_estimators=trial.suggest_int("xgb_n", 500, 2000),
            subsample=trial.suggest_float("xgb_sub", 0.5, 1.0),
            colsample_bytree=trial.suggest_float("xgb_col", 0.5, 1.0),
            random_state=SEED
        )

In [6]:
def objective(trial):
    model = get_model(trial)

    pipe = Pipeline([
        ("model", model)
    ])

    score = cross_val_score(
        pipe,
        X_train,
        y_train,
        scoring="r2",
        cv=5
    ).mean()

    return score

In [7]:
logging.set_verbosity(logging.WARNING)

def print_trial_result(study, trial):
    print(f"Trial {trial.number} | R² = {trial.value:.4f}")

study = create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=20,
    callbacks=[print_trial_result]
)

print("\nMelhor R²:", study.best_value)
print("Melhores parâmetros:", study.best_params)

Trial 0 | R² = 0.8761
Trial 1 | R² = 0.8859
Trial 2 | R² = 0.8873
Trial 3 | R² = 0.8810
Trial 4 | R² = 0.8864
Trial 5 | R² = 0.8781
Trial 6 | R² = 0.7443
Trial 7 | R² = 0.8939
Trial 8 | R² = 0.8803
Trial 9 | R² = 0.8992
Trial 10 | R² = 0.8905
Trial 11 | R² = 0.8994
Trial 12 | R² = 0.8999
Trial 13 | R² = 0.8968
Trial 14 | R² = 0.8997
Trial 15 | R² = 0.8809
Trial 16 | R² = 0.8948
Trial 17 | R² = 0.8921
Trial 18 | R² = 0.8868
Trial 19 | R² = 0.8947

Melhor R²: 0.8998844504356385
Melhores parâmetros: {'model': 'xgb', 'xgb_depth': 4, 'xgb_lr': 0.01145184722791698, 'xgb_n': 1572, 'xgb_sub': 0.6254298133033692, 'xgb_col': 0.505319590379841}


In [8]:
X_train_base, X_valid_imp, y_train_base, y_valid_imp = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=SEED
)

importance_model = XGBRegressor(
    max_depth=study.best_params["xgb_depth"],
    learning_rate=study.best_params["xgb_lr"],
    n_estimators=study.best_params["xgb_n"],
    subsample=study.best_params["xgb_sub"],
    colsample_bytree=study.best_params["xgb_col"],
    random_state=SEED
)

importance_model.fit(X_train_base, y_train_base)

result = permutation_importance(
    importance_model,
    X_valid_imp,
    y_valid_imp,
    scoring="r2",
    n_repeats=20,
    random_state=SEED
)

importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": result.importances_mean
})

importance_df = (
    importance_df
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

In [9]:
importance_df.to_csv(r"C:\Users\Gabryel\Desktop\Projetos_ML\PriceMyHome\data\processed\V1\Feature_importance.csv", index=False)